In [0]:
import pandas as pd

from sklearn.datasets import load_iris

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType
)

In [0]:
import importlib
import src.data_preparation

importlib.reload(
    src.data_preparation
)

In [0]:
from src.config import (
    RAW_TABLE,
    TRAIN_TABLE,
    TEST_TABLE,
    TEST_SIZE,
    RANDOM_STATE
)

print("RAW TABLE  :", RAW_TABLE)
print("TRAIN TABLE:", TRAIN_TABLE)
print("TEST TABLE :", TEST_TABLE)

In [0]:
from src.data_preparation import (
    load_iris_data,
    validate_data,
    remove_duplicates,
    split_data
)

In [0]:
pdf = load_iris_data()

print(
    f"Dataset loaded successfully."
)

print(
    f"Number of rows: {len(pdf)}"
)

print(
    f"Number of columns: {len(pdf.columns)}"
)

display(pdf.head())

In [0]:
print(pdf.dtypes)

In [0]:
validate_data(pdf)

In [0]:
duplicate_count = int(
    pdf.duplicated().sum()
)

print(
    f"Exact duplicate rows found: "
    f"{duplicate_count}"
)

In [0]:
pdf, removed_count = remove_duplicates(
    pdf
)

In [0]:
validate_data(pdf)

In [0]:
remaining_duplicates = int(
    pdf.duplicated().sum()
)

print(
    f"Remaining duplicate rows: "
    f"{remaining_duplicates}"
)

assert remaining_duplicates == 0, (
    "Duplicate records still exist."
)

In [0]:
print("Final row count:", len(pdf))

display(
    pdf.describe()
)

In [0]:
display(
    pdf.groupby(
        ["target", "species"]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values("target")
)

In [0]:
df = spark.createDataFrame(pdf)

print(
    f"Spark DataFrame rows: "
    f"{df.count()}"
)

display(df)

In [0]:
df.printSchema()

In [0]:
(
    df.write
      .format("delta")
      .mode("overwrite")
      .option(
          "overwriteSchema",
          "true"
      )
      .saveAsTable(RAW_TABLE)
)

print(
    f"Created Delta table: {RAW_TABLE}"
)

In [0]:
raw_df = spark.table(
    RAW_TABLE
)

print(
    f"Rows in raw table: "
    f"{raw_df.count()}"
)

display(raw_df)

In [0]:
train_pdf, test_pdf = split_data(
    pdf,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print(
    f"Train rows: {len(train_pdf)}"
)

print(
    f"Test rows : {len(test_pdf)}"
)

print(
    f"Total rows: "
    f"{len(train_pdf) + len(test_pdf)}"
)

In [0]:
assert (
    len(train_pdf) +
    len(test_pdf)
    == len(pdf)
)

print(
    "Train/Test row-count validation: PASSED"
)

In [0]:
display(
    train_pdf
    .groupby(
        ["target", "species"]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values("target")
)

In [0]:
display(
    test_pdf
    .groupby(
        ["target", "species"]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values("target")
)

In [0]:
train_df = spark.createDataFrame(
    train_pdf
)

test_df = spark.createDataFrame(
    test_pdf
)

print(
    "Train Spark rows:",
    train_df.count()
)

print(
    "Test Spark rows:",
    test_df.count()
)

In [0]:
(
    train_df.write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(TRAIN_TABLE)
)

print(
    f"Created: {TRAIN_TABLE}"
)

In [0]:
(
    test_df.write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true"
        )
        .saveAsTable(TEST_TABLE)
)

print(
    f"Created: {TEST_TABLE}"
)

In [0]:
final_raw_count = spark.table(
    RAW_TABLE
).count()

final_train_count = spark.table(
    TRAIN_TABLE
).count()

final_test_count = spark.table(
    TEST_TABLE
).count()

print(
    "=========================================="
)

print(
    f"RAW   : {final_raw_count}"
)

print(
    f"TRAIN : {final_train_count}"
)

print(
    f"TEST  : {final_test_count}"
)

print(
    "=========================================="
)

In [0]:
# Final data quality checks

assert final_raw_count > 0, "Raw table is empty."

assert (
    final_train_count + final_test_count
    == final_raw_count
), (
    f"Train + Test count mismatch: "
    f"{final_train_count} + {final_test_count} "
    f"!= {final_raw_count}"
)

assert (
    final_train_count > 0
    and final_test_count > 0
), "Train or test table is empty."

print("==========================================")
print("FINAL DATA QUALITY CHECKS: PASSED")
print(f"RAW   : {final_raw_count}")
print(f"TRAIN : {final_train_count}")
print(f"TEST  : {final_test_count}")
print("==========================================")